<a href="https://colab.research.google.com/github/Keyaki181/VinIntelligent_V1/blob/main/Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install torch


In [ ]:
!pip -q install catboost xgboost lightgbm

In [ ]:
# ============================================================
# Data-only Forecast Model
#
# The pipeline uses only fields available in the supplied CSV files and
# features derived directly from the Date column. It avoids hard-coded
# external events such as holidays, salary dates, pandemic periods, or
# manually created future campaigns. The only file written is
# `submission.csv`.
# ============================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# -----------------------------
# CONFIG
# -----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

FOLDER_PATH = "/content/drive/MyDrive/DatathonVin/"
def get_path(f):
    return os.path.join(FOLDER_PATH, f)

TEST_END = "2024-07-01"
OUTPUT_NAME = "submission.csv"
# No previous submission is read. The anchor forecast is rebuilt inside this
# script from raw CSV files so the run is reproducible.
OLD_SUBMISSION_NAME = None

# The final forecast is intentionally conservative: the internally rebuilt
# anchor is a stable baseline, while the direct/category/seasonal models add
# flexibility. A base anchor weight around 0.63 keeps the forecast from
# drifting too far when individual models disagree.
BASE_OLD_ANCHOR_WEIGHT = 0.63

# Category and direct models carry most of the signal. The seasonal component
# is kept at 20% so it stabilizes the calendar pattern without overpowering
# the transaction-driven models.
BLEND_WEIGHTS = {"category_sum": 0.44, "direct_model": 0.36, "seasonal": 0.20}

# Optional components. They are all data-derived and can be disabled for
# ablation tests without changing the rest of the pipeline.
USE_PROMO_PRIOR = True
USE_DUAL_DIRECT_MODEL = True
USE_COGS_RATIO_MODEL = True
USE_ANALOG_CORRECTION = True
USE_RATIO_RESIDUAL_MODEL = True

# Analog correction adjusts systematic errors of the seasonal baseline by
# comparing similar historical calendar/promotion-pattern buckets. The factor
# is clipped tightly to prevent a small bucket from distorting the forecast.
ANALOG_CORRECTION_STRENGTH = 0.55
RATIO_MODEL_WEIGHT = 0.42
ANALOG_FACTOR_CLIP = (0.90, 1.13)

REV_CLIP_UPPER_FACTOR = 1.75
COGS_MAX_RATIO = 0.92
SIMPLE_DIRECT_WEIGHT = 0.26

# Self-ensemble creates a few controlled variants around the same forecast.
# The base variant is largest, while the conservative/event/seasonal variants
# reduce risk on days where one source may be unreliable.
USE_SELF_ENSEMBLE_VARIANTS = True
SELF_ENSEMBLE_WEIGHTS = {
    "base": 0.42,
    "anchor_conservative": 0.22,
    "event_adaptive": 0.20,
    "seasonal_stable": 0.16,
}

# -----------------------------
# UTILS
# -----------------------------
def safe_read_csv(filename, **kwargs):
    path = get_path(filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    return pd.read_csv(path, **kwargs)

def add_calendar_features(df):
    """Calendar features derived only from the Date column.

    Cyclical encodings allow models to learn yearly, weekly and day-of-month
    patterns without hard-coding holidays, salary dates, or external events.
    """
    df = df.copy()
    d = pd.to_datetime(df["Date"])
    df["Date"] = d
    df["year"] = d.dt.year
    df["month"] = d.dt.month
    df["day"] = d.dt.day
    df["weekday"] = d.dt.weekday
    df["dayofyear"] = d.dt.dayofyear
    df["weekofyear"] = d.dt.isocalendar().week.astype(int)
    df["quarter"] = d.dt.quarter

    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df["is_month_start"] = d.dt.is_month_start.astype(int)
    df["is_month_end"] = d.dt.is_month_end.astype(int)
    df["is_quarter_end"] = d.dt.is_quarter_end.astype(int)
    df["is_double_day"] = (df["day"] == df["month"]).astype(int)

    for k in [1, 2, 3]:
        df[f"sin_y_{k}"] = np.sin(2 * np.pi * k * df["dayofyear"] / 365.25)
        df[f"cos_y_{k}"] = np.cos(2 * np.pi * k * df["dayofyear"] / 365.25)
    for k in [1, 2]:
        df[f"sin_w_{k}"] = np.sin(2 * np.pi * k * df["weekday"] / 7.0)
        df[f"cos_w_{k}"] = np.cos(2 * np.pi * k * df["weekday"] / 7.0)
    df["sin_dom"] = np.sin(2 * np.pi * df["day"] / 31.0)
    df["cos_dom"] = np.cos(2 * np.pi * df["day"] / 31.0)
    return df

def get_train_weights(dates):
    """Recency weighting based only on time order in the training data."""
    dates = pd.to_datetime(dates)
    years = dates.dt.year
    return (1.0 + (years - years.min()).clip(lower=0) * 0.055).astype(float)

def make_event_bucket(df):
    """Data-only event buckets for analog correction."""
    out = pd.Series("normal", index=df.index, dtype="object")
    if "is_weekend" in df:
        out[df["is_weekend"].fillna(0).astype(int) == 1] = "weekend"
    if "is_double_day" in df:
        out[df["is_double_day"].fillna(0).astype(int) == 1] = "double_day"
    if "promo_count" in df:
        out[df["promo_count"].fillna(0).astype(float) > 0] = "promo_pattern"
    return out

# -----------------------------
# PROMOTION FEATURES FROM PROVIDED CSV ONLY
# -----------------------------
def build_promo_pattern_features(promo, date_frame):
    """Create promotion-prior features from historical promotion rows.

    This does not create future campaign records. For any date, it measures how
    similar its month/day and month/weekday are to promotion periods observed in
    promotions.csv.
    """
    promo = promo.copy()
    promo["start_date"] = pd.to_datetime(promo["start_date"])
    promo["end_date"] = pd.to_datetime(promo["end_date"])
    rows = []
    for _, p in promo.iterrows():
        for d in pd.date_range(p["start_date"], p["end_date"], freq="D"):
            rows.append({
                "month": d.month,
                "day": d.day,
                "weekday": d.weekday(),
                "discount_value": float(p.get("discount_value", 0.0) or 0.0),
                "stackable_flag": int(p.get("stackable_flag", 0) or 0),
            })

    out = date_frame[["Date"]].copy()
    out["Date"] = pd.to_datetime(out["Date"])
    out["month"] = out["Date"].dt.month
    out["day"] = out["Date"].dt.day
    out["weekday"] = out["Date"].dt.weekday
    if not rows:
        for c in ["promo_prior_score", "promo_prior_discount", "promo_prior_count", "promo_prior_stackable"]:
            out[c] = 0.0
        return out[["Date", "promo_prior_score", "promo_prior_discount", "promo_prior_count", "promo_prior_stackable"]]

    hist = pd.DataFrame(rows)
    md = hist.groupby(["month", "day"]).agg(
        promo_count_md=("discount_value", "count"),
        promo_discount_md=("discount_value", "median"),
        promo_stackable_md=("stackable_flag", "mean"),
    ).reset_index()
    mw = hist.groupby(["month", "weekday"]).agg(
        promo_count_mw=("discount_value", "count"),
        promo_discount_mw=("discount_value", "median"),
        promo_stackable_mw=("stackable_flag", "mean"),
    ).reset_index()
    out = out.merge(md, on=["month", "day"], how="left").merge(mw, on=["month", "weekday"], how="left")
    out["promo_prior_count"] = out["promo_count_md"].fillna(0) * 0.75 + out["promo_count_mw"].fillna(0) * 0.25
    out["promo_prior_discount"] = out["promo_discount_md"].fillna(out["promo_discount_mw"]).fillna(0)
    out["promo_prior_stackable"] = out["promo_stackable_md"].fillna(out["promo_stackable_mw"]).fillna(0)
    max_count = max(float(out["promo_prior_count"].max()), 1.0)
    out["promo_prior_score"] = (out["promo_prior_count"] / max_count).clip(0, 1)
    return out[["Date", "promo_prior_score", "promo_prior_discount", "promo_prior_count", "promo_prior_stackable"]]

def build_category_promo_pattern_features(promo, categories, date_frame):
    """Category-level promotion-prior features from promotions.csv."""
    promo = promo.copy()
    promo["start_date"] = pd.to_datetime(promo["start_date"])
    promo["end_date"] = pd.to_datetime(promo["end_date"])
    cat_set = set(map(str, categories))
    rows = []
    for _, p in promo.iterrows():
        app_cat = p.get("applicable_category", np.nan)
        target_categories = [app_cat] if pd.notna(app_cat) and str(app_cat) in cat_set else categories
        for d in pd.date_range(p["start_date"], p["end_date"], freq="D"):
            for cat in target_categories:
                rows.append({
                    "month": d.month,
                    "day": d.day,
                    "weekday": d.weekday(),
                    "category": cat,
                    "discount_value": float(p.get("discount_value", 0.0) or 0.0),
                })

    base = date_frame[["Date"]].copy()
    base["Date"] = pd.to_datetime(base["Date"])
    base = base.assign(key=1).merge(pd.DataFrame({"category": categories, "key": 1}), on="key").drop(columns="key")
    base["month"] = base["Date"].dt.month
    base["day"] = base["Date"].dt.day
    base["weekday"] = base["Date"].dt.weekday
    if not rows:
        base["cat_promo_val"] = 0.0
        base["cat_promo_sum"] = 0.0
        base["cat_promo_count"] = 0.0
        return base[["Date", "category", "cat_promo_val", "cat_promo_sum", "cat_promo_count"]]

    hist = pd.DataFrame(rows)
    md = hist.groupby(["category", "month", "day"]).agg(
        cat_count_md=("discount_value", "count"),
        cat_discount_md=("discount_value", "median"),
        cat_sum_md=("discount_value", "sum"),
    ).reset_index()
    mw = hist.groupby(["category", "month", "weekday"]).agg(
        cat_count_mw=("discount_value", "count"),
        cat_discount_mw=("discount_value", "median"),
        cat_sum_mw=("discount_value", "sum"),
    ).reset_index()
    base = base.merge(md, on=["category", "month", "day"], how="left").merge(mw, on=["category", "month", "weekday"], how="left")
    base["cat_promo_count"] = base["cat_count_md"].fillna(0) * 0.75 + base["cat_count_mw"].fillna(0) * 0.25
    base["cat_promo_val"] = base["cat_discount_md"].fillna(base["cat_discount_mw"]).fillna(0)
    base["cat_promo_sum"] = base["cat_sum_md"].fillna(0) * 0.75 + base["cat_sum_mw"].fillna(0) * 0.25
    return base[["Date", "category", "cat_promo_val", "cat_promo_sum", "cat_promo_count"]]

# -----------------------------
# BASE FEATURE TABLES
# -----------------------------
def add_clean_seasonal_baselines(df):
    """Create lag and seasonal-memory features from historical sales only.

    Lag 364 keeps weekday alignment across years, 365 keeps calendar-day
    alignment, and 728 provides a two-year reference. Median seasonal values
    are used as fallbacks because they are less sensitive to isolated spikes.
    """
    df = df.copy().sort_values("Date")
    for lag in [364, 365, 728]:
        df[f"rev_lag_{lag}"] = df["Revenue"].shift(lag)
        df[f"cogs_lag_{lag}"] = df["COGS"].shift(lag)

    train = df[df["Revenue"].notna()].copy()
    clean = train.copy()
    if clean.empty:
        clean = train.copy()

    df = df.join(clean.groupby(["month", "day"])["Revenue"].median().rename("rev_md_clean"), on=["month", "day"])
    df = df.join(clean.groupby(["month", "weekday"])["Revenue"].median().rename("rev_mw_clean"), on=["month", "weekday"])
    df = df.join(clean.groupby(["month"])["Revenue"].median().rename("rev_m_clean"), on=["month"])
    df = df.join(clean.groupby(["month", "day"])["COGS"].median().rename("cogs_md_clean"), on=["month", "day"])
    df = df.join(clean.groupby(["month", "weekday"])["COGS"].median().rename("cogs_mw_clean"), on=["month", "weekday"])
    df = df.join(clean.groupby(["month"])["COGS"].median().rename("cogs_m_clean"), on=["month"])

    for col in ["rev_md_clean", "rev_mw_clean", "rev_m_clean"]:
        df[col] = df[col].fillna(train["Revenue"].median())
    for col in ["cogs_md_clean", "cogs_mw_clean", "cogs_m_clean"]:
        df[col] = df[col].fillna(train["COGS"].median())

    for lag in [364, 365, 728]:
        df[f"rev_lag_{lag}"] = df[f"rev_lag_{lag}"].fillna(df["rev_mw_clean"]).fillna(df["rev_m_clean"])
        df[f"cogs_lag_{lag}"] = df[f"cogs_lag_{lag}"].fillna(df["cogs_mw_clean"]).fillna(df["cogs_m_clean"])

    clean_base = 0.65 * df["rev_mw_clean"] + 0.35 * df["rev_md_clean"]
    target_year = df["Date"].dt.year
    seasonal_2023 = 0.72 * df["rev_lag_364"] + 0.08 * df["rev_lag_728"] + 0.20 * clean_base
    seasonal_2024 = 0.48 * df["rev_lag_364"] + 0.22 * df["rev_lag_728"] + 0.30 * clean_base
    seasonal_train = 0.72 * df["rev_lag_364"] + 0.10 * df["rev_lag_365"] + 0.18 * clean_base
    df["seasonal_rev"] = np.where(target_year == 2023, seasonal_2023, np.where(target_year == 2024, seasonal_2024, seasonal_train))

    cogs_clean_base = 0.65 * df["cogs_mw_clean"] + 0.35 * df["cogs_md_clean"]
    df["seasonal_cogs"] = 0.62 * df["cogs_lag_364"] + 0.12 * df["cogs_lag_728"] + 0.26 * cogs_clean_base
    return df

def load_base_features():
    """Build the daily feature table used by the direct and ratio models."""
    sales = safe_read_csv("sales.csv", parse_dates=["Date"]).sort_values("Date")
    traffic = safe_read_csv("web_traffic.csv", parse_dates=["date"]).rename(columns={"date": "Date"})
    promo = safe_read_csv("promotions.csv")

    df = add_calendar_features(pd.DataFrame({"Date": pd.date_range(sales["Date"].min(), TEST_END, freq="D")}))

    traffic = add_calendar_features(traffic)
    traffic_cols = [c for c in ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec"] if c in traffic.columns]
    traffic_daily = traffic.groupby("Date")[traffic_cols].mean().reset_index()
    traffic_daily = add_calendar_features(traffic_daily)

    tw = traffic_daily.groupby(["month", "weekday"])[traffic_cols].median().add_suffix("_mw").reset_index()
    tm = traffic_daily.groupby(["month", "day"])[traffic_cols].median().add_suffix("_md").reset_index()
    df = df.merge(tw, on=["month", "weekday"], how="left").merge(tm, on=["month", "day"], how="left")
    for col in traffic_cols:
        df[f"{col}_base"] = df[f"{col}_mw"].fillna(df[f"{col}_md"]).fillna(traffic_daily[col].median())
    if "sessions_base" not in df.columns:
        df["sessions_base"] = df["sessions_mw"].fillna(df["sessions_md"]).fillna(traffic_daily["sessions"].median())
    df["sess_base"] = df["sessions_base"]

    promo_prior = build_promo_pattern_features(promo, df[["Date"]])
    df = df.merge(promo_prior, on="Date", how="left")
    for c in ["promo_prior_score", "promo_prior_discount", "promo_prior_count", "promo_prior_stackable"]:
        df[c] = df[c].fillna(0)
    df["promo_val"] = df["promo_prior_discount"]
    df["promo_sum"] = df["promo_prior_discount"] * df["promo_prior_count"]
    df["promo_count"] = df["promo_prior_count"]
    df["promo_stackable"] = df["promo_prior_stackable"]
    df["traffic_intensity"] = np.log1p(df["sess_base"]) * (1 + df["promo_prior_score"])

    df = df.merge(sales[["Date", "Revenue", "COGS"]], on="Date", how="left")
    df = add_clean_seasonal_baselines(df)
    drop_cols = [c for c in df.columns if c.endswith("_mw") or c.endswith("_md")]
    return df.drop(columns=drop_cols)

def load_category_features(master_df):
    """Build category-level targets and features from transaction tables.

    The model forecasts category revenue first, then sums categories back to a
    daily total. This preserves product-mix information that is lost when only
    total revenue is modeled.
    """
    orders = safe_read_csv("orders.csv", parse_dates=["order_date"])
    items = safe_read_csv("order_items.csv")
    prods = safe_read_csv("products.csv")

    item = items.merge(prods[["product_id", "category", "cogs"]], on="product_id", how="left")
    item = item.merge(orders[["order_id", "order_date"]], on="order_id", how="left")
    item["line_revenue"] = item["quantity"] * item["unit_price"]
    item["line_cogs"] = item["quantity"] * item["cogs"]
    item["Date"] = item["order_date"]

    cat_daily = item[item["Date"].notna()].groupby(["Date", "category"]).agg(
        cat_revenue=("line_revenue", "sum"), cat_cogs=("line_cogs", "sum")
    ).reset_index()
    cat_daily = add_calendar_features(cat_daily)
    cat_daily["cat_cogs_ratio"] = (cat_daily["cat_cogs"] / cat_daily["cat_revenue"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    categories = sorted(cat_daily["category"].dropna().unique().tolist())

    cat_promo = build_category_promo_pattern_features(safe_read_csv("promotions.csv"), categories, master_df[["Date"]])

    long_df = master_df[["Date"]].assign(key=1).merge(pd.DataFrame({"category": categories, "key": 1}), on="key").drop(columns="key")
    keep_cols = [
        "Date", "month", "day", "weekday", "weekofyear", "sess_base", "promo_val", "promo_sum", "promo_count",
        "is_double_day", "is_weekend", "sin_y_1", "cos_y_1", "sin_y_2", "cos_y_2",
    ]
    keep_cols = [c for c in keep_cols if c in master_df.columns]
    long_df = long_df.merge(master_df[keep_cols], on="Date", how="left")
    long_df = long_df.merge(cat_promo, on=["Date", "category"], how="left")
    for c in ["cat_promo_val", "cat_promo_sum", "cat_promo_count"]:
        long_df[c] = long_df[c].fillna(0)
    long_df = long_df.merge(cat_daily[["Date", "category", "cat_revenue", "cat_cogs", "cat_cogs_ratio"]], on=["Date", "category"], how="left")
    long_df = long_df.sort_values(["category", "Date"])

    for lag in [364, 728]:
        long_df[f"cat_rev_lag_{lag}"] = long_df.groupby("category")["cat_revenue"].shift(lag)

    clean_c = long_df[long_df["cat_revenue"].notna()].copy()
    if clean_c.empty:
        clean_c = long_df[long_df["cat_revenue"].notna()].copy()

    long_df = long_df.join(clean_c.groupby(["category", "month", "weekday"])["cat_revenue"].median().rename("cat_rev_mw_clean"), on=["category", "month", "weekday"])
    long_df = long_df.join(clean_c.groupby(["category", "month"])["cat_revenue"].median().rename("cat_rev_m_clean"), on=["category", "month"])
    for lag in [364, 728]:
        long_df[f"cat_rev_lag_{lag}"] = long_df[f"cat_rev_lag_{lag}"].fillna(long_df["cat_rev_mw_clean"]).fillna(long_df["cat_rev_m_clean"]).fillna(0)
    long_df["cat_seasonal_rev"] = 0.70 * long_df["cat_rev_lag_364"] + 0.10 * long_df["cat_rev_lag_728"] + 0.20 * long_df["cat_rev_mw_clean"].fillna(long_df["cat_rev_m_clean"]).fillna(0)

    clean_ratio = cat_daily.copy()
    if clean_ratio.empty:
        clean_ratio = cat_daily.copy()
    long_df = long_df.join(clean_ratio.groupby(["category", "month", "weekday"])["cat_cogs_ratio"].median().rename("ratio_cmw"), on=["category", "month", "weekday"])
    long_df = long_df.join(clean_ratio.groupby(["category", "month"])["cat_cogs_ratio"].median().rename("ratio_cm"), on=["category", "month"])
    long_df = long_df.join(clean_ratio.groupby(["category"])["cat_cogs_ratio"].median().rename("ratio_c"), on=["category"])
    global_ratio = clean_ratio["cat_cogs_ratio"].replace([np.inf, -np.inf], np.nan).dropna().median()
    long_df["expected_cat_cogs_ratio"] = long_df["ratio_cmw"].fillna(long_df["ratio_cm"]).fillna(long_df["ratio_c"]).fillna(global_ratio).clip(0.25, COGS_MAX_RATIO)
    return long_df, categories

# -----------------------------
# MODELS
# -----------------------------
def train_predict_direct(master):
    """Train the main daily revenue model.

    LGBM/XGB/CatBoost are blended because their errors differ: LGBM is strong
    on tabular interactions, XGBoost is robust for absolute-error behavior, and
    CatBoost adds a smoother tree ensemble. A smaller simple model is blended
    in to reduce overfitting from the richer feature set.
    """
    train = master[master["Revenue"].notna()].copy()
    test = master[master["Revenue"].isna()].copy()

    features = [
        "month", "day", "weekday", "weekofyear", "dayofyear", "quarter",
        "is_weekend", "is_month_start", "is_month_end", "is_quarter_end",
        "is_double_day",
        "sess_base", "unique_visitors_base", "page_views_base", "bounce_rate_base", "avg_session_duration_sec_base",
        "promo_val", "promo_sum", "promo_count", "promo_stackable", "traffic_intensity",
        "rev_lag_364", "rev_lag_365", "rev_lag_728", "seasonal_rev",
        "sin_y_1", "cos_y_1", "sin_y_2", "cos_y_2", "sin_y_3", "cos_y_3",
        "sin_w_1", "cos_w_1", "sin_w_2", "cos_w_2",
    ]
    features = [f for f in features if f in master.columns]
    X = train[features].fillna(0)
    X_test = test[features].fillna(0)
    y = np.log1p(train["Revenue"])
    w = get_train_weights(train["Date"])

    m_lgb = LGBMRegressor(
        n_estimators=3400, learning_rate=0.0115, num_leaves=31, min_child_samples=25,
        subsample=0.88, colsample_bytree=0.88, reg_alpha=0.05, reg_lambda=1.7,
        importance_type="gain", verbosity=-1, random_state=SEED, n_jobs=-1,
    ).fit(X, y, sample_weight=w)
    m_xgb = XGBRegressor(
        n_estimators=2300, learning_rate=0.0135, max_depth=5, min_child_weight=4,
        subsample=0.88, colsample_bytree=0.88, reg_lambda=1.7, reg_alpha=0.03,
        objective="reg:absoluteerror", random_state=SEED, n_jobs=-1,
    ).fit(X, y, sample_weight=w)
    m_cat = CatBoostRegressor(
        iterations=2300, learning_rate=0.017, depth=6, loss_function="MAE",
        random_seed=SEED, allow_writing_files=False, verbose=0,
    ).fit(X, y, sample_weight=w)

    pred_log_rich = 0.50 * m_lgb.predict(X_test) + 0.32 * m_xgb.predict(X_test) + 0.18 * m_cat.predict(X_test)

    if USE_DUAL_DIRECT_MODEL:
        simple_features = [f for f in [
            "month", "day", "weekday", "sess_base", "promo_val", "is_double_day",
            "rev_lag_364", "seasonal_rev"
        ] if f in master.columns]
        Xs = train[simple_features].fillna(0)
        Xs_test = test[simple_features].fillna(0)
        simple_lgb = LGBMRegressor(
            n_estimators=2200, learning_rate=0.016, num_leaves=23, min_child_samples=35,
            subsample=0.90, colsample_bytree=0.90, reg_lambda=2.0,
            importance_type="gain", verbosity=-1, random_state=SEED + 7, n_jobs=-1,
        ).fit(Xs, y, sample_weight=w)
        simple_xgb = XGBRegressor(
            n_estimators=1500, learning_rate=0.018, max_depth=4, min_child_weight=5,
            subsample=0.90, colsample_bytree=0.90, reg_lambda=2.0,
            objective="reg:absoluteerror", random_state=SEED + 7, n_jobs=-1,
        ).fit(Xs, y, sample_weight=w)
        pred_log_simple = 0.68 * simple_lgb.predict(Xs_test) + 0.32 * simple_xgb.predict(Xs_test)
        pred_log = (1 - SIMPLE_DIRECT_WEIGHT) * pred_log_rich + SIMPLE_DIRECT_WEIGHT * pred_log_simple
    else:
        pred_log = pred_log_rich

    return test[["Date"]].assign(dir_revenue=np.expm1(pred_log))

def train_predict_category(cat_long):
    """Forecast revenue per product category, then aggregate by day."""
    train = cat_long[cat_long["cat_revenue"].notna()].copy()
    test = cat_long[cat_long["cat_revenue"].isna()].copy()
    features = [
        "category", "month", "day", "weekday", "weekofyear", "sess_base",
        "promo_val", "promo_sum", "promo_count",
        "cat_promo_val", "cat_promo_sum", "cat_promo_count",
        "is_double_day",
        "cat_rev_lag_364", "cat_rev_lag_728", "cat_seasonal_rev",
        "sin_y_1", "cos_y_1", "sin_y_2", "cos_y_2",
    ]
    features = [f for f in features if f in cat_long.columns]
    X = train[features].fillna(0)
    X_test = test[features].fillna(0)
    y = np.log1p(train["cat_revenue"].fillna(0))
    w = get_train_weights(train["Date"])

    model = CatBoostRegressor(
        iterations=3300, learning_rate=0.0175, depth=6, loss_function="MAE",
        cat_features=["category"], random_seed=SEED, allow_writing_files=False, verbose=0,
    ).fit(X, y, sample_weight=w)
    pred = np.expm1(model.predict(X_test))
    # Keep category model tied to category seasonality; avoids wild category mix errors.
    test["pred_cat_rev"] = np.maximum(0.82 * pred + 0.18 * test["cat_seasonal_rev"].fillna(0).values, 0)
    test["pred_cat_cogs"] = test["pred_cat_rev"] * test["expected_cat_cogs_ratio"].fillna(0.75)
    return test.groupby("Date").agg(
        category_sum_revenue=("pred_cat_rev", "sum"),
        category_sum_cogs=("pred_cat_cogs", "sum"),
    ).reset_index()

def train_predict_cogs_ratio(master):
    """Predict COGS as a ratio of revenue rather than as an independent level.

    This keeps COGS economically consistent with Revenue and makes the forecast
    less sensitive to daily revenue scale errors.
    """
    train = master[(master["Revenue"].notna()) & (master["Revenue"] > 0)].copy()
    test = master[master["Revenue"].isna()].copy()
    ratio = (train["COGS"] / train["Revenue"]).replace([np.inf, -np.inf], np.nan)
    ratio = ratio.clip(max(0.20, ratio.quantile(0.01)), min(COGS_MAX_RATIO, ratio.quantile(0.99))).fillna(ratio.median())

    features = [f for f in [
        "month", "day", "weekday", "weekofyear", "dayofyear", "quarter", "is_weekend", "is_month_end",
        "is_double_day",
        "sess_base", "promo_val", "traffic_intensity",
        "rev_lag_364", "rev_lag_728", "seasonal_rev", "cogs_lag_364", "cogs_lag_728", "seasonal_cogs",
        "sin_y_1", "cos_y_1", "sin_y_2", "cos_y_2",
    ] if f in master.columns]
    X = train[features].fillna(0)
    X_test = test[features].fillna(0)
    w = get_train_weights(train["Date"])

    model = LGBMRegressor(
        n_estimators=1900, learning_rate=0.012, num_leaves=15, min_child_samples=35,
        subsample=0.88, colsample_bytree=0.88, reg_alpha=0.05, reg_lambda=2.0,
        objective="mae", verbosity=-1, random_state=SEED + 13, n_jobs=-1,
    ).fit(X, ratio, sample_weight=w)
    pred_ratio = model.predict(X_test)
    seasonal_ratio = (test["seasonal_cogs"] / test["seasonal_rev"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    seasonal_ratio = seasonal_ratio.fillna(ratio.median())
    return test[["Date"]].assign(pred_cogs_ratio=(0.72 * pred_ratio + 0.28 * seasonal_ratio).clip(0.25, COGS_MAX_RATIO))

# -----------------------------
# META / ANALOG CORRECTION
# -----------------------------
def build_bucket_ratio(master):
    train = master[(master["Revenue"].notna()) & (master["seasonal_rev"] > 0)].copy()
    clean = train.copy()
    if clean.empty:
        clean = train.copy()
    clean["event_bucket"] = make_event_bucket(clean)
    clean["ratio_actual_to_seasonal"] = (clean["Revenue"] / clean["seasonal_rev"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    clean["ratio_actual_to_seasonal"] = clean["ratio_actual_to_seasonal"].clip(0.55, 1.75)
    return clean

def attach_analog_factor(test, master):
    test = test.copy()
    test["event_bucket"] = make_event_bucket(test)
    clean = build_bucket_ratio(master)
    test["analog_ratio"] = np.nan
    test["analog_count"] = 0.0

    group_plans = [
        ["event_bucket", "month", "weekday"],
        ["event_bucket", "month"],
        ["month", "weekday"],
        ["month"],
    ]
    for i, keys in enumerate(group_plans):
        grp = clean.groupby(keys)["ratio_actual_to_seasonal"].agg(["median", "count"]).reset_index()
        grp = grp.rename(columns={"median": f"ratio_{i}", "count": f"count_{i}"})
        test = test.merge(grp, on=keys, how="left")
        mask = test["analog_ratio"].isna() & test[f"ratio_{i}"].notna()
        test.loc[mask, "analog_ratio"] = test.loc[mask, f"ratio_{i}"]
        test.loc[mask, "analog_count"] = test.loc[mask, f"count_{i}"]
        test = test.drop(columns=[f"ratio_{i}", f"count_{i}"])

    test["analog_ratio"] = test["analog_ratio"].fillna(1.0).clip(0.70, 1.35)
    # Shrink correction when analog bucket has few historical samples.
    conf = (test["analog_count"].fillna(0) / 14.0).clip(0, 1)
    raw_factor = 1.0 + ANALOG_CORRECTION_STRENGTH * conf * (test["analog_ratio"] - 1.0)
    test["analog_factor_bucket"] = raw_factor.clip(*ANALOG_FACTOR_CLIP)
    return test

def train_predict_ratio_residual(master):
    """Learn a smooth multiplier for Revenue / seasonal_rev.

    The seasonal baseline often gets the calendar shape right but misses the
    level. This model predicts a limited correction factor rather than a full
    replacement forecast.
    """
    train = master[(master["Revenue"].notna()) & (master["seasonal_rev"] > 0)].copy()
    test = master[master["Revenue"].isna()].copy()

    y_ratio = (train["Revenue"] / train["seasonal_rev"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    y_ratio = y_ratio.clip(0.55, 1.75).fillna(1.0)
    y = np.log(y_ratio)

    features = [f for f in [
        "year", "month", "day", "weekday", "weekofyear", "dayofyear", "quarter",
        "is_weekend", "is_month_start", "is_month_end", "is_double_day",

        "sess_base", "unique_visitors_base", "page_views_base", "bounce_rate_base", "avg_session_duration_sec_base",
        "promo_val", "promo_sum", "promo_count", "promo_stackable", "traffic_intensity",
        "rev_lag_364", "rev_lag_365", "rev_lag_728", "seasonal_rev",
        "sin_y_1", "cos_y_1", "sin_y_2", "cos_y_2", "sin_y_3", "cos_y_3",
        "sin_w_1", "cos_w_1", "sin_w_2", "cos_w_2",
    ] if f in master.columns]

    X = train[features].fillna(0)
    X_test = test[features].fillna(0)
    w = get_train_weights(train["Date"])

    model = LGBMRegressor(
        n_estimators=1600, learning_rate=0.014, num_leaves=17, min_child_samples=35,
        subsample=0.88, colsample_bytree=0.88, reg_alpha=0.08, reg_lambda=2.2,
        objective="mae", verbosity=-1, random_state=SEED + 31, n_jobs=-1,
    ).fit(X, y, sample_weight=w)
    ratio = np.exp(model.predict(X_test)).clip(0.86, 1.17)
    return test[["Date"]].assign(ratio_model_factor=ratio)

def compute_anchor_weight(df, base_w=BASE_OLD_ANCHOR_WEIGHT):
    """Dynamic anchor weight using only model disagreement and data-derived signals."""
    w = np.full(len(df), base_w, dtype=float)
    year = df["Date"].dt.year
    w += np.where(year == 2024, 0.012, 0.0)

    rel_gap = np.abs(df["Revenue_New"] - df["R_old"]) / (np.abs(df["R_old"]) + 1)
    w += np.where(rel_gap > 0.28, 0.018, 0.0)
    w += np.where(rel_gap > 0.45, 0.020, 0.0)

    gap_new = np.abs(df["Revenue_New"] - df["seasonal_rev"]) / (np.abs(df["seasonal_rev"]) + 1)
    gap_old = np.abs(df["R_old"] - df["seasonal_rev"]) / (np.abs(df["seasonal_rev"]) + 1)
    w -= np.where((gap_new < 0.12) & (gap_old > 0.25), 0.016, 0.0)

    if "promo_count" in df:
        w -= np.where(df["promo_count"].fillna(0).values > 0, 0.010, 0.0)
    return np.clip(w, 0.54, 0.70)

# -----------------------------
# INTERNAL ULTIMATE ANCHOR
# -----------------------------
def generate_internal_ultimate_anchor():
    """Build a stable anchor forecast from raw CSV files only.

    This anchor is deliberately simpler than the main model: it uses calendar
    features, traffic, returns, and category mix. It acts as a conservative
    reference forecast in the final blend, reducing the chance that the richer
    models overreact to noisy features.

    Returns columns: Date, R_old, C_old.
    """

    def _cat_mix_features():
        items = safe_read_csv("order_items.csv")
        prods = safe_read_csv("products.csv")
        orders = safe_read_csv("orders.csv", parse_dates=["order_date"])

        if "subtotal" not in items.columns:
            if "unit_price" in items.columns and "quantity" in items.columns:
                items["subtotal"] = items["unit_price"] * items["quantity"]
            else:
                for col in ["total_price", "price", "amount"]:
                    if col in items.columns:
                        items["subtotal"] = items[col]
                        break
        if "subtotal" not in items.columns:
            items["subtotal"] = 0.0

        x = items.merge(prods[["product_id", "category"]], on="product_id", how="left")
        x = x.merge(orders[["order_id", "order_date"]], on="order_id", how="left")
        daily_cat_rev = x.groupby(["order_date", "category"])["subtotal"].sum().unstack(fill_value=0)
        daily_total = daily_cat_rev.sum(axis=1).replace(0, np.nan)
        cat_mix = daily_cat_rev.div(daily_total, axis=0).reset_index().fillna(0)
        cat_mix = cat_mix.rename(columns={"order_date": "Date"})
        return cat_mix

    def _add_anchor_calendar(df):
        df = df.copy()
        df["month"] = df["Date"].dt.month
        df["day"] = df["Date"].dt.day
        df["weekday"] = df["Date"].dt.weekday
        df["is_double_day"] = (df["day"] == df["month"]).astype(int)
        for k in [1, 2]:
            df[f"sin_{k}"] = np.sin(2 * np.pi * k * df["Date"].dt.dayofyear / 365.25)
            df[f"cos_{k}"] = np.cos(2 * np.pi * k * df["Date"].dt.dayofyear / 365.25)
        return df

    def _robust_merge(master_df, source, value_cols):
        source = source.copy()
        source["Date"] = pd.to_datetime(source["Date"])
        source["m"] = source["Date"].dt.month
        source["d"] = source["Date"].dt.day
        seasonal = source.groupby(["m", "d"])[value_cols].mean().reset_index()
        out = master_df.merge(source[["Date"] + value_cols], on="Date", how="left")
        out["m"] = out["Date"].dt.month
        out["d"] = out["Date"].dt.day
        out = out.merge(seasonal, on=["m", "d"], how="left", suffixes=("", "_avg"))
        for col in value_cols:
            out[col] = out[col].fillna(out[col + "_avg"])
        return out.drop(columns=[c for c in out.columns if c.endswith("_avg") or c in ["m", "d"]])

    sales = safe_read_csv("sales.csv", parse_dates=["Date"]).sort_values("Date")
    promo = safe_read_csv("promotions.csv")
    traffic = safe_read_csv("web_traffic.csv", parse_dates=["date"]).rename(columns={"date": "Date"})
    returns = safe_read_csv("returns.csv", parse_dates=["return_date"]).rename(columns={"return_date": "Date"})

    full_range = pd.date_range(start=sales["Date"].min(), end=TEST_END, freq="D")
    df_anchor = pd.DataFrame({"Date": full_range})

    cat_mix = _cat_mix_features()
    df_anchor = df_anchor.merge(cat_mix, on="Date", how="left")
    cat_cols = [c for c in cat_mix.columns if c != "Date"]
    df_anchor = _add_anchor_calendar(df_anchor)

    df_anchor = _robust_merge(df_anchor, traffic, ["sessions", "unique_visitors"])
    ret_daily = returns.groupby("Date")["refund_amount"].sum().reset_index()
    df_anchor = _robust_merge(df_anchor, ret_daily, ["refund_amount"])

    df_anchor["promo_val"] = 0.0
    for _, p in promo.iterrows():
        mask = (df_anchor["Date"] >= pd.to_datetime(p["start_date"])) & (df_anchor["Date"] <= pd.to_datetime(p["end_date"]))
        df_anchor.loc[mask, "promo_val"] = p["discount_value"]

    df_anchor["traffic_intensity"] = df_anchor["sessions"] * (df_anchor["promo_val"] + 1)

    for col in cat_cols:
        df_anchor[col] = df_anchor.groupby("month")[col].transform(lambda x: x.fillna(x.mean()))
    df_anchor[cat_cols] = df_anchor[cat_cols].fillna(0)

    df_anchor["sample_weight"] = 1.0 + (df_anchor["Date"].dt.year - 2012) * 0.15
    df_anchor = df_anchor.merge(sales[["Date", "Revenue", "COGS"]], on="Date", how="left")

    features_rev = [
        "month", "day", "weekday", "is_double_day", "promo_val", "sessions",
        "traffic_intensity"
    ] + cat_cols + [c for c in df_anchor.columns if c.startswith("sin_")]
    features_rev = [c for c in dict.fromkeys(features_rev) if c in df_anchor.columns]

    train_mask = df_anchor["Revenue"].notna()
    test_mask = df_anchor["Revenue"].isna()
    X_train = df_anchor.loc[train_mask, features_rev].fillna(0)
    X_test = df_anchor.loc[test_mask, features_rev].fillna(0)
    y_rev_log = np.log1p(df_anchor.loc[train_mask, "Revenue"])
    weights = df_anchor.loc[train_mask, "sample_weight"]

    model_cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, loss_function="MAE", depth=7, random_seed=0, allow_writing_files=False, verbose=0)
    model_xgb = XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=6, objective="reg:absoluteerror", random_state=0, n_jobs=-1)
    model_cat.fit(X_train, y_rev_log, sample_weight=weights)
    model_xgb.fit(X_train, y_rev_log, sample_weight=weights)

    final_log_rev = 0.7 * model_cat.predict(X_test) + 0.3 * model_xgb.predict(X_test)
    df_anchor.loc[test_mask, "Revenue"] = np.expm1(final_log_rev)

    df_anchor.loc[train_mask, "Pred_Rev_Log"] = y_rev_log
    df_anchor.loc[test_mask, "Pred_Rev_Log"] = final_log_rev
    features_cogs = features_rev + ["Pred_Rev_Log", "refund_amount"]
    y_ratio = np.log1p(df_anchor.loc[train_mask, "COGS"] / (df_anchor.loc[train_mask, "Revenue"] + 1))

    model_cogs = CatBoostRegressor(iterations=2000, learning_rate=0.03, loss_function="MAE", random_seed=0, allow_writing_files=False, verbose=0)
    model_cogs.fit(df_anchor.loc[train_mask, features_cogs].fillna(0), y_ratio)
    df_anchor.loc[test_mask, "COGS"] = df_anchor.loc[test_mask, "Revenue"] * np.expm1(model_cogs.predict(df_anchor.loc[test_mask, features_cogs].fillna(0)))

    df_anchor["Revenue"] = df_anchor["Revenue"].clip(0, df_anchor.loc[train_mask, "Revenue"].max() * 2)
    df_anchor["COGS"] = df_anchor["COGS"].clip(0, df_anchor["Revenue"])

    return df_anchor.loc[test_mask, ["Date", "Revenue", "COGS"]].rename(columns={"Revenue": "R_old", "COGS": "C_old"}).reset_index(drop=True)

# -----------------------------
# SELF-ENSEMBLE HELPERS
# -----------------------------
def _month_stabilize_variant(df, rev_col, cogs_col, old_weight=0.52, seasonal_weight=0.48, strength=0.26, clip=(0.945, 1.065)):
    """Lightly align a variant's monthly total to anchor/seasonal consensus.

    The correction is intentionally small and clipped. It fixes broad level
    drift while preserving daily peaks and valleys learned by the models.
    """
    out = df.copy()
    ym = out["Date"].dt.to_period("M").astype(str)
    target = out.groupby(ym)["R_old"].transform("sum") * old_weight + out.groupby(ym)["seasonal_rev"].transform("sum") * seasonal_weight
    pred = out.groupby(ym)[rev_col].transform("sum") + 1.0
    scale = (target / pred).clip(clip[0], clip[1])
    factor = (1 - strength) + strength * scale
    out[rev_col] = out[rev_col] * factor
    out[cogs_col] = out[cogs_col] * factor
    return out[[rev_col, cogs_col]]

def _make_anchor_variant(df, w, rev_new=None, cogs_new=None, name="variant", month_old=0.52, month_seas=0.48, month_strength=0.26, month_clip=(0.945, 1.065)):
    """Create one controlled forecast variant from the main model and anchor."""
    tmp = df.copy()
    if rev_new is None:
        rev_new = tmp["Revenue_New"]
    if cogs_new is None:
        cogs_new = tmp["COGS_New"]
    tmp[f"rev_{name}"] = (1 - w) * rev_new + w * tmp["R_old"]
    tmp[f"cogs_{name}"] = (1 - w) * cogs_new + w * tmp["C_old"]
    fixed = _month_stabilize_variant(
        tmp, f"rev_{name}", f"cogs_{name}",
        old_weight=month_old, seasonal_weight=month_seas,
        strength=month_strength, clip=month_clip,
    )
    return fixed[f"rev_{name}"].values, fixed[f"cogs_{name}"].values

def _weighted_positive_blend(arr, weights):
    """Blend positive forecasts robustly.

    Arithmetic blending preserves level, log blending reduces the impact of
    extreme spikes, and median shrink protects against large disagreement among
    variants.
    """
    arr = np.asarray(arr, dtype=float)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, 0, None)
    arith = arr.dot(weights)
    logmean = np.expm1(np.log1p(arr).dot(weights))
    median = np.median(arr, axis=1)
    disagreement = (np.max(arr, axis=1) - np.min(arr, axis=1)) / (median + 1.0)
    blend = 0.52 * logmean + 0.48 * arith
    # When variants disagree a lot, shrink toward the cross-variant median.
    blend = np.where(disagreement > 0.18, 0.70 * blend + 0.30 * median, blend)
    blend = np.where(disagreement > 0.32, 0.55 * blend + 0.45 * median, blend)
    return blend

def build_self_ensemble(final):
    """Create several forecast variants and blend them.

    Each variant emphasizes a different source: the main forecast, the internal
    anchor, promotion-pattern dates, or the seasonal baseline. The final blend
    is more stable than relying on any single variant.
    """
    df = final.copy()
    w_base = compute_anchor_weight(df, BASE_OLD_ANCHOR_WEIGHT)

    # Variant 1: balanced base forecast.
    rev_base, cogs_base = _make_anchor_variant(
        df, w_base, name="base",
        month_old=0.52, month_seas=0.48, month_strength=0.26, month_clip=(0.945, 1.065),
    )

    # Variant 2: more conservative anchor, useful for far-horizon dates.
    w_cons = np.clip(w_base + 0.030 + np.where(df["Date"].dt.year.values == 2024, 0.010, 0.0), 0.56, 0.73)
    rev_cons, cogs_cons = _make_anchor_variant(
        df, w_cons, name="cons",
        month_old=0.60, month_seas=0.40, month_strength=0.22, month_clip=(0.955, 1.055),
    )

    # Variant 3: trust the current model slightly more on dates that look like
    # historical promotion patterns, based only on promotions.csv.
    promo_signal = (
        df.get("promo_prior_score", pd.Series(0, index=df.index)).fillna(0).values
        if "promo_prior_score" in df.columns
        else (df.get("promo_count", pd.Series(0, index=df.index)).fillna(0).values > 0).astype(float)
    )
    event_boost = 0.025 * np.clip(promo_signal, 0, 1)
    w_event = np.clip(w_base - event_boost, 0.49, 0.69)
    rev_event, cogs_event = _make_anchor_variant(
        df, w_event, name="event",
        month_old=0.47, month_seas=0.53, month_strength=0.30, month_clip=(0.935, 1.075),
    )

    # Variant 4: seasonal-stable forecast, reducing overreaction of the main model.
    rev_new_stable = 0.84 * df["Revenue_New"].values + 0.16 * df["seasonal_rev"].values
    cogs_new_stable = 0.82 * df["COGS_New"].values + 0.18 * df["seasonal_cogs"].values
    w_stable = np.clip(w_base + 0.005, 0.54, 0.71)
    rev_stable, cogs_stable = _make_anchor_variant(
        df, w_stable, rev_new=rev_new_stable, cogs_new=cogs_new_stable, name="stable",
        month_old=0.50, month_seas=0.50, month_strength=0.24, month_clip=(0.950, 1.060),
    )

    weights = np.array([
        SELF_ENSEMBLE_WEIGHTS["base"],
        SELF_ENSEMBLE_WEIGHTS["anchor_conservative"],
        SELF_ENSEMBLE_WEIGHTS["event_adaptive"],
        SELF_ENSEMBLE_WEIGHTS["seasonal_stable"],
    ], dtype=float)

    rev_arr = np.vstack([rev_base, rev_cons, rev_event, rev_stable]).T
    rev_final = _weighted_positive_blend(rev_arr, weights)

    # Blend COGS mostly through ratios to avoid COGS/Revenue inconsistency.
    cogs_arr = np.vstack([cogs_base, cogs_cons, cogs_event, cogs_stable]).T
    ratio_arr = cogs_arr / (rev_arr + 1.0)
    ratio_arr = np.clip(np.nan_to_num(ratio_arr, nan=0.75, posinf=0.75, neginf=0.75), 0.20, COGS_MAX_RATIO)
    ratio_blend = ratio_arr.dot(weights / weights.sum())
    cogs_direct = _weighted_positive_blend(cogs_arr, weights)
    cogs_ratio = rev_final * ratio_blend
    cogs_final = 0.35 * cogs_direct + 0.65 * cogs_ratio
    return rev_final, cogs_final

# -----------------------------
# FINALIZATION
# -----------------------------
def finalize_predictions(master, direct_pred, cat_pred, ratio_pred=None, ratio_model_pred=None):
    """Combine all model outputs and prepare the final submission table."""
    test = master[master["Revenue"].isna()].copy()
    test = test.merge(direct_pred, on="Date", how="left").merge(cat_pred, on="Date", how="left")
    if ratio_pred is not None:
        test = test.merge(ratio_pred, on="Date", how="left")
    if ratio_model_pred is not None:
        test = test.merge(ratio_model_pred, on="Date", how="left")

    test["pred_cat_rev"] = test["category_sum_revenue"].fillna(test["dir_revenue"])
    test["pred_cat_cogs"] = test["category_sum_cogs"].fillna(test["category_sum_revenue"].fillna(test["dir_revenue"]) * 0.75)

    test["Revenue_New"] = (
        BLEND_WEIGHTS["category_sum"] * test["pred_cat_rev"] +
        BLEND_WEIGHTS["direct_model"] * test["dir_revenue"] +
        BLEND_WEIGHTS["seasonal"] * test["seasonal_rev"]
    )

    # Analog correction fixes systematic seasonal under/over-shoot by bucket.
    if USE_ANALOG_CORRECTION:
        test = attach_analog_factor(test, master)
        bucket_factor = test["analog_factor_bucket"].fillna(1.0).values
        if USE_RATIO_RESIDUAL_MODEL and "ratio_model_factor" in test.columns:
            model_factor = test["ratio_model_factor"].fillna(1.0).clip(*ANALOG_FACTOR_CLIP).values
            factor = (1 - RATIO_MODEL_WEIGHT) * bucket_factor + RATIO_MODEL_WEIGHT * model_factor
        else:
            factor = bucket_factor
        factor = np.clip(factor, ANALOG_FACTOR_CLIP[0], ANALOG_FACTOR_CLIP[1])
        test["Revenue_New"] = test["Revenue_New"] * factor
        test["meta_factor"] = factor
    else:
        test["meta_factor"] = 1.0

    train = master[master["Revenue"].notna()].copy()
    global_ratio = (train["COGS"] / train["Revenue"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna().median()
    if USE_COGS_RATIO_MODEL and "pred_cogs_ratio" in test.columns:
        test["pred_cogs_ratio"] = test["pred_cogs_ratio"].fillna(global_ratio).clip(0.25, COGS_MAX_RATIO)
        test["COGS_New"] = (
            0.45 * test["pred_cat_cogs"] * test["meta_factor"] +
            0.18 * test["seasonal_cogs"] * test["meta_factor"] +
            0.15 * (test["Revenue_New"] * global_ratio) +
            0.22 * (test["Revenue_New"] * test["pred_cogs_ratio"])
        )
    else:
        test["COGS_New"] = 0.56 * test["pred_cat_cogs"] * test["meta_factor"] + 0.22 * test["seasonal_cogs"] * test["meta_factor"] + 0.22 * (test["Revenue_New"] * global_ratio)

    # Build the anchor internally; no previous submission file is read.
    old = generate_internal_ultimate_anchor()
    final = test.merge(old, on="Date", how="left")
    final["R_old"] = final["R_old"].fillna(final["Revenue_New"])
    final["C_old"] = final["C_old"].fillna(final["COGS_New"])

    if USE_SELF_ENSEMBLE_VARIANTS:
        final["Revenue"], final["COGS"] = build_self_ensemble(final)
    else:
        w = compute_anchor_weight(final, BASE_OLD_ANCHOR_WEIGHT)
        final["Revenue"] = (1 - w) * final["Revenue_New"] + w * final["R_old"]
        final["COGS"] = (1 - w) * final["COGS_New"] + w * final["C_old"]

        # Month-level stabilization: keep total monthly level close to old/seasonal mix, but very lightly.
        final["ym"] = final["Date"].dt.to_period("M").astype(str)
        target_month_sum = final.groupby("ym")["R_old"].transform("sum") * 0.52 + final.groupby("ym")["seasonal_rev"].transform("sum") * 0.48
        pred_month_sum = final.groupby("ym")["Revenue"].transform("sum") + 1
        scale = (target_month_sum / pred_month_sum).clip(0.945, 1.065)
        final["Revenue"] = final["Revenue"] * (0.74 + 0.26 * scale)
        final["COGS"] = final["COGS"] * (0.74 + 0.26 * scale)

    final["Revenue"] = final["Revenue"].clip(lower=0, upper=train["Revenue"].max() * REV_CLIP_UPPER_FACTOR)
    final["COGS"] = np.minimum(final["COGS"].clip(lower=0), final["Revenue"] * COGS_MAX_RATIO)

    sample_path = get_path("sample_submission.csv")
    if os.path.exists(sample_path):
        out = pd.read_csv(sample_path, parse_dates=["Date"])[["Date"]].merge(final, on="Date", how="left")[["Date", "Revenue", "COGS"]]
    else:
        out = final.sort_values("Date")[["Date", "Revenue", "COGS"]]
    out["Date"] = pd.to_datetime(out["Date"]).dt.strftime("%Y-%m-%d")
    out["Revenue"] = out["Revenue"].fillna(method="ffill").fillna(method="bfill").clip(lower=0)
    out["COGS"] = np.minimum(out["COGS"].fillna(out["Revenue"] * global_ratio).clip(lower=0), out["Revenue"] * COGS_MAX_RATIO)
    return out

def run_model():
    """Run the complete pipeline and write only submission.csv."""
    master = load_base_features()
    cat_long, _ = load_category_features(master)
    direct_pred = train_predict_direct(master)
    cat_pred = train_predict_category(cat_long)
    ratio_pred = train_predict_cogs_ratio(master) if USE_COGS_RATIO_MODEL else None
    ratio_model_pred = train_predict_ratio_residual(master) if USE_RATIO_RESIDUAL_MODEL else None
    out = finalize_predictions(master, direct_pred, cat_pred, ratio_pred, ratio_model_pred)
    out.to_csv(get_path(OUTPUT_NAME), index=False)

if __name__ == "__main__":
    run_model()

